In [0]:
# Configuration (keep this private)
# NOTE: Replace API_KEY with your own key from openweathermap.org
# Never commit real API keys to public repositories

API_KEY = "YOUR_OPENWEATHERMAP_API_KEY_HERE"
BASE_URL = "https://api.openweathermap.org/data/2.5/weather"

CITIES = ["Mumbai", "Delhi", "Pune", "Bangalore", "Chennai",
          "Hyderabad", "Kolkata", "Ahmedabad", "Jaipur", "Surat"]

In [0]:
#Install dependencies
%pip install requests

In [0]:
# Fetch weather data from API
import requests
import json
from datetime import datetime, timezone

def fetch_weather(city: str) -> dict:
    params = {
        "q": city,
        "appid": API_KEY,
        "units": "metric"   # gives Celsius
    }
    response = requests.get(BASE_URL, params=params)
    
    if response.status_code == 200:
        data = response.json()
        data["ingested_at"] = datetime.now(timezone.utc).isoformat()
        return data
    else:
        print(f"Failed for {city}: {response.status_code} - {response.text}")
        return None

# Test with one city first
test = fetch_weather("Pune")
print(json.dumps(test, indent=2))

In [0]:
# Fetch all cities and create a DataFrame
from pyspark.sql import Row

raw_records = []
for city in CITIES:
    result = fetch_weather(city)
    if result:
        raw_records.append(Row(
            city        = result.get("name", city),
            raw_json    = json.dumps(result),
            ingested_at = result["ingested_at"]
        ))
        print(f"✓ Fetched: {city}")
    else:
        print(f"✗ Skipped: {city}")

print(f"\nTotal records fetched: {len(raw_records)}")

In [0]:
# Cell 5 — Bronze write with accidental re-run protection
from pyspark.sql.functions import col, max as spark_max
from datetime import datetime, timezone, timedelta

def ingested_in_last_n_minutes(table_name, minutes=10):
    try:
        if not spark.catalog.tableExists(table_name):
            return False
        last = spark.table(table_name) \
                    .agg(spark_max("ingested_at").alias("last")) \
                    .collect()[0]["last"]
        if last is None:
            return False
        last_utc = last.replace(tzinfo=timezone.utc)
        return datetime.now(timezone.utc) - last_utc < timedelta(minutes=minutes)
    except:
        return False

if ingested_in_last_n_minutes("weather_bronze_raw", minutes=10):
    print("⏭ Skipped — already ran within last 10 mins. Re-run intentionally? Change minutes=10 to 0.")
else:
    df = spark.createDataFrame(raw_records)
    df.write.format("delta").mode("append").saveAsTable("weather_bronze_raw")
    print(f"✓ Bronze: {len(raw_records)} records written at {datetime.now(timezone.utc).isoformat()}")